# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

# My Rule

The goal of this baseline is to rank website content by refresh priority using simple business rules instead of machine learning.

The score is based on three observable signals:

- Higher priority if the page is old.
- Higher priority if CTR is lower than expected.
- Higher priority if the page receives many impressions.

The baseline acts as a decision-support tool for editors.

It does not predict the future.
It only prioritizes pages that appear to be good refresh candidates.

## Reason Codes

STALE_CONTENT
The content has not been updated for a long time.

LOW_CTR
The page receives fewer clicks than expected.

HIGH_IMPRESSIONS
The page already receives many impressions, so improving it could have a larger impact.

REFRESH_PRIORITY
Multiple signals indicate that this page should be refreshed.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [11]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/content_refresh_anonymized.csv")

In [13]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [15]:
df["baseline_score"] = (
    (df["days_since_last_update"] * 0.35) +
    (df["content_age_days"] * 0.20) +
    ((100 - df["ctr"]) * 0.25) +
    (np.log1p(df["impressions_90d"]) * 5)
)

In [16]:
def get_reason(row):
    if row["days_since_last_update"] > 180:
        return "STALE_CONTENT"
    elif row["ctr"] < 1.5:
        return "LOW_CTR"
    elif row["avg_position"] > 10:
        return "POSITION_RISK"
    else:
        return "HIGH_IMPRESSIONS"

In [17]:
def get_action(score):
    if score >= 150:
        return "Refresh Immediately"
    elif score >= 100:
        return "Review Soon"
    else:
        return "Monitor"

In [18]:
df["reason_code"] = df.apply(get_reason, axis=1)
df["action"] = df["baseline_score"].apply(get_action)

df = df.sort_values("baseline_score", ascending=False)

In [19]:
import os

os.makedirs("work/outputs", exist_ok=True)

df.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [20]:
top20 = df.head(20)[[
    "content_id",
    "baseline_score",
    "action",
    "reason_code"
]]

top20

,content_id,baseline_score,action,reason_code
26242,content_55a5b1c46474,248.267595,Refresh Immediately,STALE_CONTENT
29384,content_f6fdf87348f6,235.643061,Refresh Immediately,STALE_CONTENT
24216,content_1b4ec72dafd4,235.093061,Refresh Immediately,STALE_CONTENT
6653,content_5fe46e04994d,234.550911,Refresh Immediately,LOW_CTR
18440,content_8d56efff1e71,233.065736,Refresh Immediately,STALE_CONTENT
6962,content_f01216059a6a,228.138960,Refresh Immediately,STALE_CONTENT
8631,content_e2b702f4f92b,225.869936,Refresh Immediately,STALE_CONTENT
15790,content_6476d1d8c050,225.751559,Refresh Immediately,STALE_CONTENT
26840,content_7f116ae1f6f5,224.753557,Refresh Immediately,STALE_CONTENT
7452,content_72496874f806,224.048702,Refresh Immediately,STALE_CONTENT


# 3. Top-20 Review

## Review Criteria

Each of the top 20 content items was reviewed based on:

- **Action** recommended by the baseline rule
- **Reason code** explaining why the page was prioritized
- **Confidence** in the recommendation
- **What would make the recommendation wrong**

---

| Rank | Content ID | Action | Reason Code | Confidence | What Would Make It Wrong |
|------|------------|--------|-------------|------------|--------------------------|
| 1 | content_55a5b1c46474 | Refresh Immediately | STALE_CONTENT | High | The content was recently updated but the dataset has not yet reflected the changes. |
| 2 | content_f6fdf87348f6 | Refresh Immediately | STALE_CONTENT | High | The page remains evergreen and still performs well despite its age. |
| 3 | content_1b4ec72dafd4 | Refresh Immediately | STALE_CONTENT | High | Search demand has naturally declined rather than the content becoming outdated. |
| 4 | content_5fe46e04994d | Refresh Immediately | LOW_CTR | High | Low CTR is caused by SERP features or search intent rather than poor content quality. |
| 5 | content_8d56efff1e71 | Refresh Immediately | STALE_CONTENT | High | A recent refresh has not yet been indexed by search engines. |
| 6 | content_f01216059a6a | Refresh Immediately | STALE_CONTENT | High | The page continues to satisfy user intent despite being old. |
| 7 | content_e2b702f4f92b | Refresh Immediately | STALE_CONTENT | High | Performance remains stable after reviewing longer historical trends. |
| 8 | content_6476d1d8c050 | Refresh Immediately | STALE_CONTENT | High | Traffic decline is due to seasonality instead of outdated content. |
| 9 | content_7f116ae1f6f5 | Refresh Immediately | STALE_CONTENT | High | The content targets evergreen topics that rarely require updates. |
| 10 | content_72496874f806 | Refresh Immediately | STALE_CONTENT | High | Google ranking fluctuations recover naturally without requiring a refresh. |
| 11 | content_02b0d6e30129 | Refresh Immediately | STALE_CONTENT | High | Future data shows traffic recovering without intervention. |
| 12 | content_fb7fb643eff7 | Refresh Immediately | LOW_CTR | Medium | CTR is temporarily reduced because of changing search result layouts. |
| 13 | content_e6955a2c59dc | Refresh Immediately | LOW_CTR | Medium | Low CTR is expected for this type of search query. |
| 14 | content_d25a099b3726 | Refresh Immediately | STALE_CONTENT | High | The page still delivers strong engagement despite its age. |
| 15 | content_1210e6c2e909 | Refresh Immediately | LOW_CTR | Medium | Competitor changes, not content quality, caused the CTR decline. |
| 16 | content_06e19c6486b0 | Refresh Immediately | STALE_CONTENT | High | The page has already been refreshed outside the available dataset. |
| 17 | content_da2adacc2e93 | Refresh Immediately | LOW_CTR | Medium | The title or meta description has recently been updated but indexing is incomplete. |
| 18 | content_7a888d3d99c8 | Refresh Immediately | STALE_CONTENT | High | User behavior has shifted temporarily rather than permanently. |
| 19 | content_612eeab9024b | Refresh Immediately | LOW_CTR | Medium | CTR improves after SERP changes without requiring content updates. |
| 20 | content_df1fa766cac2 | Refresh Immediately | STALE_CONTENT | High | Additional business metrics may show the page is still valuable despite its score. |

---

## Overall Observation

Most of the highest-ranked pages were flagged because of the **STALE_CONTENT** reason code, indicating that content age was the strongest contributor to the baseline score. Several pages were also prioritized due to **LOW_CTR**, suggesting that click-through performance is another important indicator of refresh priority. Since this is a rule-based baseline rather than a learned model, these recommendations should be treated as **decision-support** and reviewed by an editor before action is taken.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

# 4. Weak Picks + Leakage Check

## Weak Picks

Although the baseline rule identifies high-priority content for refresh, some recommendations may be incorrect.

Possible weak picks include:

- Evergreen pages that continue to perform well despite being old.
- Seasonal content whose traffic naturally rises and falls during the year.
- Pages with low CTR because of SERP features (such as AI Overviews or Featured Snippets) rather than outdated content.
- High-impression pages that receive a high score mainly because of traffic volume instead of actual content decay.

These pages should be manually reviewed before any content refresh decision is made.

---

## Leakage Check

The baseline score was built only from features that are available at the decision time.

**Features used:**
- `days_since_last_update`
- `content_age_days`
- `ctr`
- `impressions_90d`

**Features deliberately excluded:**
- `trend_direction`
- `trend_pct`

These excluded fields are derived from future performance and would introduce data leakage if used as model features.

Therefore, the baseline rule does not use future-window information or label-derived columns, making it an honest decision-support baseline.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.